## BSplineの定義

In [ ]:
import numpy as np
from scipy.interpolate import BSpline

# set parameter (t, c, k)
k = 4
t = np.array([0, 0, 0, 0, 0, 0.5, 1, 1, 1, 1, 1], dtype=float)
c = np.array([
    [0.0, 0.0],   # c0
    [0.5, 1.2],   # c1
    [1.5, 0.4],   # c2
    [2.0, 2.2],   # c3
    [3.0, -0.6],  # c4
    [4.0, 0.0],   # c5
], dtype=float)

# define bspline curve
bspline = BSpline(t, c, k)

In [ ]:
print('t:', bspline.t)
print('c:', bspline.c)
print('k:', bspline.k)

t, c, k = bspline.tck
print('t:', t)
print('c:', c)
print('k:', k)

## BSpline曲線の取得

In [ ]:

# evaluate
u_min = np.min(bspline.t)
u_max = np.max(bspline.t)
u = np.linspace(u_min, u_max, 101)
curve = bspline(u)   # shape = (len(u), 2)

# plot
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(c[:, 0], c[:, 1], 'o:', markerfacecolor='none', color='k', label='$\mathbf{p}_{i}$')
ax.plot(curve[:, 0], curve[:, 1], label='$\mathbf{c}(u)$')
ax.legend()
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print('u.shape:\t', u.shape)
print('curve.shape:\t', curve.shape)

## 基底関数の取得

In [ ]:
import numpy as np
from scipy.interpolate import BSpline
np.set_printoptions(formatter={'float': '{:8.3f}'.format}, linewidth=999)

# ===== 共通パラメータ =====
k = 4
t = np.array([0, 0, 0, 0, 0, 0.5, 1, 1, 1, 1, 1], dtype=float)
l = 5
x = np.linspace(0, 1, l)
m = len(t)
n = m - (k + 1)

print('t:', t)
print('x:', x)
print('k:', k, '\tl:', l, '\tm:', m, '\tn:', n)

### BSpline.design_matrix(x, t, k)

In [ ]:
basis_matrix = BSpline.design_matrix(x, t, k).toarray()

print(basis_matrix)

In [ ]:
print('basis_matrix.shape: ', basis_matrix.shape)
print(f'(l, n) = ({l}, {n})')

### BSpline.basis_element(t_internal, extrapolate=False)

In [ ]:
basis_matrix = np.zeros((len(x), n))

x[-1] -= (1e-10)
for j in range(n):
    t_internal = t[j:j + k + 2]
    basis_func = BSpline.basis_element(t_internal, extrapolate=False)
    basis_matrix[:, j] = basis_func(x)

print(basis_matrix)


### cに単位行列を与える方法

In [ ]:
# 基底関数行列を格納
basis_matrix = np.zeros((len(x), n))
for j in range(n):
    c = np.zeros(n)
    c[j] = 1.0
    bspline = BSpline(t, c, k)
    basis_matrix[:, j] = bspline(x)

print(basis_matrix)


In [ ]:
c = np.eye(n, n)
bspline = BSpline(t, c, k)
basis_matrix = bspline(x)

print(basis_matrix)

## 微分の取得

In [ ]:
import numpy as np
from scipy.interpolate import BSpline

# set parameter (t, c, k)
k = 4
t = np.array([0, 0, 0, 0, 0, 0.5, 0.5, 0.5, 1, 1, 1, 1, 1], dtype=float)
c = np.array([
    [0.0, 0.0],   # c0
    [0.5, 1.2],   # c1
    [1.0, 2.5],   # c2
    [1.5, 0.4],   # c3
    [2.0, 2.2],   # c4
    [2.5, 1.8],   # c5
    [3.0, -0.6],  # c6
    [4.0, 0.0],   # c7
], dtype=float)
l = 101
u = np.linspace(np.min(t), np.max(t), l)
m = len(t)
n = m - (k + 1)

print('t:', t)
print('k:', k, '\tl:', l, '\tm:', m, '\tn:', n)

### BSpline.derivative(nu=1)

In [ ]:
r=1
bspline = BSpline(t, c, k)
bspline_deriv = bspline.derivative(nu=r)
deriv = bspline_deriv(u)
print(deriv)

In [ ]:
# plot
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(figsize=(3 * k, 3 * 2), ncols=k, nrows=2, squeeze=False)
axis_label = ['x', 'y', 'z']

bspline = BSpline(t, c, k)
for i in range(2):
    for r in range(1, k + 1):
        ax = axes[i, r - 1]
        t_unique = np.array(sorted(set(t.tolist())))
        for j, (t0, t1) in enumerate(zip(t_unique[:-1], t_unique[1:])):
            try:
                bspline_deriv = bspline.derivative(nu=r)
                u = t0 + 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, 22)[:-1])) * (t1 - t0)
                deriv = bspline_deriv(u)
                ax.plot(u, deriv[:, i], color=cm.tab10(j % 10))
            except:
                print(f'ValueError: The spline has internal repeated knots and is not differentiable {r} times')
                pass

        for knot in t:
            ax.axvline(knot, linestyle=':', color='k', alpha=0.1)
        
        ax.set_xlabel('$u$')
        ax.set_ylabel('derivative')
        ax.set_title(rf'$\frac{{d^{r}{axis_label[i]}}}{{du^{{{r}}}}}$')
        ax.grid(True)

plt.tight_layout()
plt.show()

### BSpline.\_\_call\_\_(x, nu=0, extrapolate=None)

In [ ]:
r=1
bspline = BSpline(t, c, k)
deriv = bspline(u, nu=r)

In [ ]:
# plot
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(figsize=(3 * k, 3 * 2), ncols=k, nrows=2, squeeze=False)
axis_label = ['x', 'y', 'z']

for i in range(2):
    for r in range(1, k + 1):
        ax = axes[i, r - 1]
        t_unique = np.array(sorted(set(t.tolist())))
        
        for j, (t0, t1) in enumerate(zip(t_unique[:-1], t_unique[1:])):
            u = t0 + 0.5 * (1 - np.cos(np.pi * np.linspace(0, 1, 22)[:-1])) * (t1 - t0)
            deriv = bspline(u, nu=r)
            ax.plot(u, deriv[:, i], color=cm.tab10(j % 10))

        for knot in t:
            ax.axvline(knot, linestyle=':', color='k', alpha=0.1)
        
        ax.set_xlabel('$u$')
        ax.set_ylabel('derivative')
        ax.set_title(rf'$\frac{{d^{r}{axis_label[i]}}}{{du^{{{r}}}}}$')
        ax.grid(True)

plt.tight_layout()
plt.show()

## ノットの挿入

In [ ]:
# # plot
# import matplotlib.pyplot as plt
# import matplotlib.cm as cm

# # evaluate
# u_min = np.min(bspline.t)
# u_max = np.max(bspline.t)
# u = np.linspace(u_min, u_max, 101)

# for r in range(1, k+1):
#     print(r)
#     fig, ax = plt.subplots(figsize=(4, 3))

#     c = np.eye(n, n)
#     bspline = BSpline(t, c, k)
#     spline_deriv = bspline.derivative(nu=r)  # r 階微分スプラインを生成
#     basis_matrix_deriv_1 = spline_deriv(u)
#     for deriv in basis_matrix_deriv_1.T:
#         ax.plot(u, deriv, label=f'$\\frac{{d^{nu}x}}{{du^{nu}}}$')
#     ax.legend()
#     ax.set_xlabel('x')
#     ax.set_ylabel('y')
#     # ax.set_aspect('equal')
#     ax.grid(True)
#     plt.tight_layout()
#     plt.show()

In [ ]:
# plot
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# evaluate
u_min = np.min(bspline.t)
u_max = np.max(bspline.t)
u = np.linspace(u_min, u_max, 101)

t_unique = np.array(sorted(set(t.tolist())))

for r in range(1, k+1):

    c = np.eye(n, n)
    bspline = BSpline(t, c, k)

    fig, ax = plt.subplots(figsize=(4+1, 3))
    for j, (t0, t1) in enumerate(zip(t_unique[:-1], t_unique[1:])):
        
        u = np.linspace(t0, t1, 999)[:-1]
        basis_matrix = bspline(u, nu=r)

        for i, deriv in enumerate(basis_matrix.T):
            color = cm.tab10(i % 10)
            label = (f'$N_{{{i}, {k}}}^{r}(u)$' if r != 0 else f'$N_{{{i}, {k}}}(u)$') if t1 == t[-(k + 1)] else None
            ax.plot(u, deriv, label=label)
    
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    # ax.set_aspect('equal')
    ax.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# c = np.eye(n, n)
# bspline = BSpline(t, c, k)
# spline_deriv = bspline.derivative(nu=r)  # r 階微分スプラインを生成
# basis_matrix_deriv_1 = spline_deriv(u)

# print("① 単位行列方式（r 階微分）")
# print(basis_matrix_deriv_1)
